# HotpotQA Evaluation with Critic System

This notebook demonstrates how to evaluate the Critic System on the stratified 300-question HotpotQA dataset.

**Requirements:** 7.1, 7.2, 7.3

## Overview

The evaluation process:
1. Load the stratified 300-question HotpotQA dataset
2. Process each question through the Critic System
3. Collect answers in HotpotQA evaluation format
4. Calculate accuracy metrics
5. Compare to 60.17% baseline
6. Track iteration counts and validation outcomes

## Setup

In [ ]:
import sys
import json
import logging
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.critic.evaluate_hotpotqa import evaluate_hotpotqa, load_hotpotqa_dataset

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

print("✓ Imports successful")

## Load Dataset

First, let's load and inspect the HotpotQA dataset.

In [ ]:
# Path to the stratified 300-question dataset
dataset_path = project_root / "Dataset" / "hotpot_eval_300.json"

# Load dataset
dataset = load_hotpotqa_dataset(str(dataset_path))

print(f"Loaded {len(dataset)} questions")
print(f"\nExample question:")
print(json.dumps(dataset[0], indent=2))

## Setup RAG Pipeline Components

Configure your retriever and LLM for the evaluation.

**Note:** Replace these with your actual RAG pipeline components.

In [ ]:
# TODO: Replace with your actual RAG pipeline components
#
# Example for LangChain:
#
# from langchain_community.vectorstores import Chroma
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# from langchain.retrievers import ContextualCompressionRetriever
# from langchain.retrievers.document_compressors import CohereRerank
#
# # Load vector store
# vectorstore = Chroma(
#     persist_directory="path/to/vectorstore",
#     embedding_function=OpenAIEmbeddings()
# )
#
# # Create retriever with reranking
# base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
# compressor = CohereRerank(top_n=5)
# retriever = ContextualCompressionRetriever(
#     base_compressor=compressor,
#     base_retriever=base_retriever
# )
#
# # Create LLM
# llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# For demonstration, using mock components
print("⚠️  Using mock components for demonstration")
print("   Replace with your actual RAG pipeline components")

def mock_retriever(query: str):
    """Mock retriever that returns empty results."""
    return []

class MockLLM:
    """Mock LLM that returns placeholder answers."""
    def invoke(self, prompt: str):
        return "I do not know."

retriever = mock_retriever
llm = MockLLM()

print("✓ RAG components configured")

## Load Baseline Answers (Optional)

If you have baseline answers from a previous evaluation, load them here.

In [ ]:
# Load baseline answers if available
baseline_answers = None
baseline_answers_path = project_root / "evaluation_results" / "baseline_answers.json"

if baseline_answers_path.exists():
    print(f"Loading baseline answers from: {baseline_answers_path}")
    with open(baseline_answers_path, 'r') as f:
        baseline_answers = json.load(f)
    print(f"✓ Loaded {len(baseline_answers)} baseline answers")
else:
    print("No baseline answers found")
    print("Baseline answers will be generated during evaluation")

## Run Small-Scale Test (10 questions)

Before running the full evaluation, test on a small subset to verify everything works.

In [ ]:
# Create small test dataset
test_dataset = dataset[:10]
test_dataset_path = project_root / "Dataset" / "hotpot_eval_test_10.json"

with open(test_dataset_path, 'w') as f:
    json.dump(test_dataset, f, indent=2)

print(f"Created test dataset with {len(test_dataset)} questions")
print(f"Saved to: {test_dataset_path}")

In [ ]:
# Run evaluation on test dataset
print("Running evaluation on test dataset...\n")

test_metrics = evaluate_hotpotqa(
    dataset_path=str(test_dataset_path),
    retriever=retriever,
    llm=llm,
    baseline_answers=baseline_answers,
    output_dir=str(project_root / "evaluation_results" / "test"),
    enable_logging=True
)

print("\n" + "=" * 80)
print("TEST RESULTS")
print("=" * 80)
print(f"Accuracy: {test_metrics.accuracy:.2f}%")
print(f"Correct: {test_metrics.correct_answers}/{test_metrics.total_questions}")
print(f"Processing time: {test_metrics.processing_time:.2f} seconds")

## Run Full Evaluation (300 questions)

Now run the complete evaluation on all 300 questions.

**Note:** This may take a while depending on your LLM and retrieval speed.

In [ ]:
# Run full evaluation
print("Running full evaluation on 300 questions...\n")
print("This may take a while...\n")

metrics = evaluate_hotpotqa(
    dataset_path=str(dataset_path),
    retriever=retriever,
    llm=llm,
    baseline_answers=baseline_answers,
    output_dir=str(project_root / "evaluation_results"),
    enable_logging=True
)

## Display Results

In [ ]:
print("=" * 80)
print("EVALUATION RESULTS")
print("=" * 80)
print()
print(f"Total questions:       {metrics.total_questions}")
print(f"Correct answers:       {metrics.correct_answers}")
print(f"Accuracy:              {metrics.accuracy:.2f}%")
print(f"Baseline accuracy:     {metrics.baseline_accuracy:.2f}%")
print(f"Improvement:           {metrics.accuracy_improvement:+.2f}%")
print()
print(f"Processing time:       {metrics.processing_time:.2f} seconds")
print(f"Avg time per question: {metrics.processing_time / metrics.total_questions:.2f} seconds")
print()

if metrics.accuracy > metrics.baseline_accuracy:
    print(f"✓ SUCCESS: Critic system improved accuracy by {metrics.accuracy_improvement:.2f}%")
elif metrics.accuracy == metrics.baseline_accuracy:
    print(f"= NEUTRAL: Critic system maintained baseline accuracy")
else:
    print(f"✗ REGRESSION: Critic system decreased accuracy by {metrics.accuracy_improvement:.2f}%")

## Analyze Results

Load and analyze the detailed results.

In [ ]:
# Find the most recent results file
results_dir = project_root / "evaluation_results"
results_files = sorted(results_dir.glob("detailed_results_*.json"))

if results_files:
    latest_results_file = results_files[-1]
    print(f"Loading results from: {latest_results_file}")
    
    with open(latest_results_file, 'r') as f:
        detailed_results = json.load(f)
    
    # Analyze incorrect answers
    incorrect = [r for r in detailed_results if not r['is_correct']]
    
    print(f"\nIncorrect answers: {len(incorrect)}")
    print(f"\nFirst 5 incorrect answers:")
    for i, result in enumerate(incorrect[:5], 1):
        print(f"\n{i}. Question: {result['question']}")
        print(f"   Gold answer: {result['gold_answer']}")
        print(f"   Final answer: {result['final_answer']}")
else:
    print("No results files found")

## Visualize Results (Optional)

Create visualizations of the evaluation results.

In [ ]:
import matplotlib.pyplot as plt

# Accuracy comparison
fig, ax = plt.subplots(figsize=(10, 6))

systems = ['Baseline', 'Critic System']
accuracies = [metrics.baseline_accuracy, metrics.accuracy]
colors = ['#3498db', '#2ecc71' if metrics.accuracy > metrics.baseline_accuracy else '#e74c3c']

bars = ax.bar(systems, accuracies, color=colors, alpha=0.7)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('HotpotQA Evaluation: Accuracy Comparison', fontsize=14, fontweight='bold')
ax.set_ylim(0, 100)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.2f}%',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Improvement: {metrics.accuracy_improvement:+.2f}%")

## Summary

The evaluation is complete! Results have been saved to the `evaluation_results` directory.

### Next Steps

1. Review detailed results in `evaluation_results/detailed_results_*.json`
2. Analyze incorrect answers to identify failure patterns
3. Adjust critic system parameters if needed
4. Re-run evaluation to measure improvements

### Files Generated

- `detailed_results_*.json` - Full results for each question
- `metrics_summary_*.json` - Overall metrics and statistics